# Introduction to Retrieval Augmented Generation with S&P 500 news

In this notebook, you will explore how to build a simple Retrieval-Augmented Generation (RAG) pipeline using financial news articles from S&P 500 companies.

We'll start by vectorizing text data, creating a vector store using FAISS, and integrating it with OpenAI's GPT models to answer questions using retrieved information.

This workflow emulates real-world systems in finance where natural language data (news, filings, analyst reports) are used to support decision-making.

# 📌 Objectives

By the end of this notebook, students will be able to:

1. **Perform Semantic Search with Metadata Filtering:**
   - Query the provided FAISS vector store to retrieve relevant financial news articles based on natural language questions.
   - Apply optional filters using metadata such as ticker or publication date to refine search results.

2. **Enrich Data with Company Metadata:**
   - Use the `yfinance` library to retrieve company-level metadata (company name, sector, industry) for tickers in the dataset.
   - Integrate this metadata to support enhanced filtering and analysis of news data.

3. **Build a Retrieval-Augmented Generation (RAG) Pipeline:**
   - Combine retrieved news snippets as context to generate answers using OpenAI’s GPT models.
   - Construct effective prompts that guide the language model to provide concise, context-aware responses.

4. **Evaluate and Analyze RAG Outputs:**
   - Review generated answers alongside the supporting news excerpts.
   - Reflect on the strengths and limitations of the simple RAG pipeline and consider potential improvements, such as adding more filters or refining retrieval strategies.

5. **Incorporate Financial Metadata into Retrieval Context:**
   - Enrich retrieved news snippets with key financial metadata including ticker, company name, sector, and industry.
   - Format prompts that combine both text excerpts and metadata to provide richer context to the language model.

6. **Generate Context-Aware Answers Using OpenAI Models:**
   - Construct and send prompts to an LLM that leverage both news content and metadata to produce concise, informed financial analysis.

7. **Compare Answers With and Without Metadata:**
   - Evaluate the impact of including financial metadata on answer quality using criteria such as clarity, detail, accuracy, and contextual relevance.
   - Summarize findings to reflect on the role of metadata in improving retrieval-augmented generation.

> ### ▶️ How to run this notebook
> Run the cells **in order, from the top**. The RAG v2 section redefines the retrieval
> filters used by the pipelines, so running the v2 cells out of order (or re-running only
> part of the section) will produce results that do not match the analysis below.
> The Yahoo Finance step takes a few minutes; everything after it is fast apart from the
> OpenAI calls (≈ 32 requests to `gpt-4o-mini` in total).

## Install and Import important librairies

First, we install and import the necessary libraries for:
- Text embedding generation (sentence-transformers)
- Efficient similarity search (faiss)
- Data manipulation (pandas, numpy)
- Visualization (matplotlib)

> ℹ️ FAISS uses inner product for cosine similarity by normalizing vectors.

In [1]:
%pip install sentence-transformers
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.0 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import matplotlib.pyplot as plt
import faiss

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load news data
We load a CSV file of financial news, focusing on TITLE and SUMMARY, along with metadata like TICKER and PUBLICATION_DATE.
These will be embedded into vectors and used for semantic retrieval.

In [5]:
K = 25

In [6]:
df_news = pd.read_csv('/content/drive/MyDrive/Maestria de Inteligencia Artificial Aplicada/Fintech and Digital Innovation in Finance/Semana 5/df_news.csv')
df_news['PUBLICATION_DATE'] = pd.to_datetime(df_news['PUBLICATION_DATE']).dt.date
display(df_news)

,TICKER,TITLE,SUMMARY,PUBLICATION_DATE,PROVIDER,URL
0,MMM,2 Dow Jones Stocks with Promising Prospects an...,The Dow Jones (^DJI) is made up of 30 of the m...,2025-05-29,StockStory,https://finance.yahoo.com/news/2-dow-jones-sto...
1,MMM,3 S&P 500 Stocks Skating on Thin Ice,The S&P 500 (^GSPC) is often seen as a benchma...,2025-05-27,StockStory,https://finance.yahoo.com/news/3-p-500-stocks-...
2,MMM,3M Rises 15.8% YTD: Should You Buy the Stock N...,"MMM is making strides in the aerospace, indust...",2025-05-22,Zacks,https://finance.yahoo.com/news/3m-rises-15-8-y...
3,MMM,Q1 Earnings Roundup: 3M (NYSE:MMM) And The Res...,Quarterly earnings results are a good time to ...,2025-05-22,StockStory,https://finance.yahoo.com/news/q1-earnings-rou...
4,MMM,3 Cash-Producing Stocks with Questionable Fund...,While strong cash flow is a key indicator of s...,2025-05-19,StockStory,https://finance.yahoo.com/news/3-cash-producin...
...,...,...,...,...,...,...
4866,ZTS,2 Dividend Stocks to Buy With $500 and Hold Fo...,Zoetis is a leading animal health company with...,2025-05-23,Motley Fool,https://www.fool.com/investing/2025/05/23/2-di...
4867,ZTS,Zoetis (NYSE:ZTS) Declares US$0.50 Dividend Pe...,Zoetis (NYSE:ZTS) recently affirmed a dividend...,2025-05-22,Simply Wall St.,https://finance.yahoo.com/news/zoetis-nyse-zts...
4868,ZTS,Jim Cramer on Zoetis (ZTS): “It Does Seem to B...,We recently published a list of Jim Cramer Tal...,2025-05-21,Insider Monkey,https://finance.yahoo.com/news/jim-cramer-zoet...
4869,ZTS,Zoetis (ZTS) Upgraded to Buy: Here's Why,Zoetis (ZTS) might move higher on growing opti...,2025-05-21,Zacks,https://finance.yahoo.com/news/zoetis-zts-upgr...


In [7]:
df_news['EMBEDDED_TEXT'] = df_news['TITLE'] + ' : ' + df_news['SUMMARY']

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Implement FAISS vector store
We:
- Use a pre-trained sentence transformer (all-MiniLM-L6-v2) to embed documents.
- Normalize vectors to use cosine similarity.
- Create a FAISS index and implement a basic search function.

This will allow us to retrieve relevant news snippets given a natural language question.


In [9]:
# Load model and compute embeddings
text_embeddings = model.encode(df_news['EMBEDDED_TEXT'].tolist(), convert_to_numpy=True)

# Normalize embeddings to use cosine similarity (via inner product in FAISS)
text_embeddings = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)

# Prepare metadata
documents = df_news['EMBEDDED_TEXT'].tolist()
metadata = [
    {
        'PUBLICATION_DATE': row['PUBLICATION_DATE'],
        'TICKER': row['TICKER'],
        'PROVIDER': row['PROVIDER']
    }
    for _, row in df_news.iterrows()
]

In [10]:
embedding_dim = text_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)  # Cosine similarity via inner product
faiss_index.add(text_embeddings)

In [11]:
class FaissVectorStore:
    def __init__(self, model, index, embeddings, documents, metadata):
        self.model = model
        self.index = index
        self.embeddings = embeddings
        self.documents = documents
        self.metadata = metadata

    def search(self, query, k=5, metadata_filter=None):
        query_embedding = self.model.encode([query])
        query_embedding = query_embedding / np.linalg.norm(query_embedding)

        if metadata_filter:
            filtered_indices = [i for i, meta in enumerate(self.metadata) if metadata_filter(meta)]
            if not filtered_indices:
                return []
            filtered_embeddings = self.embeddings[filtered_indices]
            temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
            temp_index.add(filtered_embeddings)
            D, I = temp_index.search(query_embedding, k)
            indices = [filtered_indices[i] for i in I[0]]
        else:
            D, I = self.index.search(query_embedding, k)
            indices = I[0]
            D = D[0]

        results = []
        for idx, sim in zip(indices, D):
            results.append((self.documents[idx], self.metadata[idx], float(sim)))
        return results

In [12]:
# Create FAISS-based store
faiss_store = FaissVectorStore(
    model=model,
    index=faiss_index,
    embeddings=text_embeddings,
    documents=documents,
    metadata=metadata
)

### Setup OpenAI Client

👉 **Instructions**:
- Import the `OpenAI` client from the `openai` Python library.
- You will need an **OpenAI API key** to use their models programmatically:
  - Go to [https://platform.openai.com/](https://platform.openai.com/) and sign up or log in.
  - Create an API key from your [API keys dashboard](https://platform.openai.com/account/api-keys).
  - ⚠️ **Keep your API key private** and **do not** share or hardcode it in public notebooks.
- Note that **usage of the OpenAI API is not free**. You will need to:
  - Add a payment method.
  - Monitor your usage to avoid unexpected charges.
  - Optionally set usage limits from your account settings.
- You can refer to the **course’s Study Resources** for a step-by-step guide on creating an OpenAI account and retrieving your API key.

Then:
- Initialize the client with `OpenAI(api_key="YOUR_KEY_HERE")`.
- Send a test request using `.responses.create()` and the `"gpt-4o-mini"` model with a simple prompt:

  ```python
  response = client.responses.create(
      model="gpt-4o-mini",
      input="Write a one-sentence bedtime story about a unicorn."
  )
  print(response.output_text)


In [13]:
from openai import OpenAI
import os, getpass

# SECURITY: never hard-code your API key inside the notebook.
# The key is read from an environment variable; if it is missing you are
# prompted for it, and the value is never stored in the .ipynb file.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Test request using the Responses API, as specified in the instructions.
response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a one-sentence bedtime story about a unicorn."
)
print(response.output_text)

# The rest of the notebook uses the Chat Completions API, because the RAG
# pipelines need multi-message prompts (system + user) and a temperature
# setting. Both endpoints call the same model.
test = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
)
print("Chat Completions endpoint:", test.choices[0].message.content)

Enter your OpenAI API key: ··········
As the moonlight danced over the quiet forest, Luna the unicorn whispered her dreams to the stars, knowing that with a sprinkle of magic, anything was possible by morning.
Chat Completions endpoint: ready


## Retrieve Additional Metadata from Yahoo Finance

👉 **Instructions**:
- We will enrich our news dataset by retrieving **company-level metadata** using the `yfinance` library.
- The goal is to map each unique stock ticker (`TICKER`) in the dataset to:
  - `COMPANY_NAME`
  - `SECTOR`
  - `INDUSTRY`

> ℹ️ `yfinance` fetches live data from Yahoo Finance. If you're running this in a cloud environment or during peak hours, expect some tickers to fail or rate limits to apply.

✅ After this step, you will have a new DataFrame (e.g. `df_meta`) with the columns `TICKER`, `COMPANY_NAME`, `SECTOR`, `INDUSTRY` that maps tickers to their company names, sectors, and industries. This metadata will be useful later to add filters and analysis based on sector or industry categories.


In [14]:
%pip install yfinance
import yfinance as yf
import time

# Get unique tickers from the news DataFrame
unique_tickers = df_news['TICKER'].unique()
print(f"Tickers to resolve: {len(unique_tickers)}")

records, failed, no_name = [], [], []

for ticker_symbol in unique_tickers:
    company_name = sector = industry = 'N/A'
    try:
        info = yf.Ticker(ticker_symbol).info or {}

        # Yahoo sometimes returns a valid sector/industry but no longName,
        # which would silently break any company-name lookup later on.
        company_name = info.get('longName') or info.get('shortName') or 'N/A'
        sector       = info.get('sector')   or 'N/A'
        industry     = info.get('industry') or 'N/A'

        if company_name == 'N/A':
            no_name.append(ticker_symbol)
        if sector == 'N/A' and industry == 'N/A':
            failed.append(ticker_symbol)

    except Exception as e:
        # Keep the ticker with N/A values instead of dropping it, so that
        # df_meta stays aligned with the tickers present in df_news.
        failed.append(ticker_symbol)
        print(f"Could not fetch data for {ticker_symbol}: {type(e).__name__}")
        time.sleep(0.2)

    records.append({
        'TICKER': ticker_symbol,
        'COMPANY_NAME': company_name,
        'SECTOR': sector,
        'INDUSTRY': industry,
    })

df_meta = pd.DataFrame(records)

# ---- Data-quality report ------------------------------------------------
print(f"\nRows in df_meta: {len(df_meta)} (one per ticker in df_news)")
print(f"Missing COMPANY_NAME : {(df_meta['COMPANY_NAME'] == 'N/A').sum()}  -> {no_name}")
print(f"Missing SECTOR       : {(df_meta['SECTOR'] == 'N/A').sum()}")
print(f"Missing INDUSTRY     : {(df_meta['INDUSTRY'] == 'N/A').sum()}")
print(f"Fully unresolved     : {len(failed)}  -> {failed}")
print("\nNOTE: tickers without COMPANY_NAME cannot be matched by company name in "
      "the RAG v2 filters below; they remain reachable by ticker symbol.")
display(df_meta.head())

Tickers to resolve: 490


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BK"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CTRA"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DAY"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IPG"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}
ERROR:yfinance:


Rows in df_meta: 490 (one per ticker in df_news)
Missing COMPANY_NAME : 11  -> ['ANSS', 'BK', 'CTRA', 'DAY', 'FI', 'HOLX', 'IPG', 'JNPR', 'K', 'MMC', 'WBA']
Missing SECTOR       : 11
Missing INDUSTRY     : 11
Fully unresolved     : 11  -> ['ANSS', 'BK', 'CTRA', 'DAY', 'FI', 'HOLX', 'IPG', 'JNPR', 'K', 'MMC', 'WBA']

NOTE: tickers without COMPANY_NAME cannot be matched by company name in the RAG v2 filters below; they remain reachable by ticker symbol.


,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
0,MMM,3M Company,Industrials,Conglomerates
1,AOS,A. O. Smith Corporation,Industrials,Specialty Industrial Machinery
2,ABT,Abbott Laboratories,Healthcare,Medical Devices
3,ABBV,AbbVie Inc.,Healthcare,Drug Manufacturers - General
4,ACN,Accenture plc,Technology,Information Technology Services


## Retrieval-Augmented Generation (RAG): Retrieve Documents and Generate Answers

👉 **Instructions**:

In this part of the assignment, your task is to build a simple Retrieval-Augmented Generation (RAG) pipeline that:

- Takes a user question as input.
- Searches the FAISS vector store to find a set of relevant financial news articles based on semantic similarity.
- Uses the retrieved news articles as context to generate a clear, concise answer to the question by interacting with the OpenAI language model.
- Returns both the generated answer and the underlying news snippets used for context.

### What you need to focus on:

- Implement a retrieval mechanism to query your vector store and obtain the top relevant documents for any question.
- Construct prompts that effectively combine retrieved news content with the user’s question to guide the language model’s response.
- Use the OpenAI API to generate answers grounded in the retrieved context.
- Organize the outputs so that for each question, you have:
  - The generated answer.
  - The collection of news excerpts used to produce that answer.

### What you will be provided:

- Helper functions to display outputs in markdown format.
- Lists of example questions covering topics, companies, and industries to test your implementation.

---

Your solution can take any form or structure you find appropriate, as long as it fulfills these core objectives. This exercise will give you hands-on experience with integrating retrieval and generation for practical applications in finance.


#### Print markdown
You can use the following function to print answers from GPT4o-mini in markdown.

In [15]:
from IPython.display import Markdown, display

def print_markdown(text):
    display(Markdown(text))

#### Predefined questions

In [16]:
from IPython.display import Markdown, display

def print_markdown(text):
    display(Markdown(text))

questions_topic = [
"What are the major concerns expressed in financial news about inflation?",
"How is investor sentiment described in recent financial headlines?",
"What role is artificial intelligence playing in recent finance-related news stories?"
]

questions_company = [
"How is Microsoft being portrayed in news stories about artificial intelligence?",
"What financial news headlines connect Amazon with automation or logistics?"
]

questions_industry = [
"What are the main themes emerging in financial news about the semiconductor industry?",
"What trends are being reported in the retail industry?",
"What risks or challenges are discussed in recent news about the energy industry?"
]

def simple_rag_pipeline(query, k=K):
    """
    A simple RAG pipeline to retrieve documents and generate an answer.

    Args:
        query (str): The user's question.
        k (int): The number of top relevant documents to retrieve.

    Returns:
        tuple: A tuple containing the generated answer (str) and the list of
               retrieved news excerpts (list of str).
    """
    # 1. Retrieve relevant documents
    retrieved_results = faiss_store.search(query, k=k)
    retrieved_documents = [doc for doc, _, _ in retrieved_results]

    if not retrieved_documents:
        return "No relevant documents found to answer your question.", []

    # 2. Construct prompt for the LLM
    context = "\n\n".join(retrieved_documents)
    prompt_messages = [
        {
            "role": "system",
            "content": "You are a helpful financial assistant. Answer the user's question based ONLY on the provided context. If the answer is not in the context, politely state that you don't have enough information."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
        }
    ]

    # 3. Generate answer using OpenAI LLM
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=prompt_messages,
            temperature=0.1 # Keep temperature low for factual answers
        )
        generated_answer = response.choices[0].message.content
    except Exception as e:
        generated_answer = f"An error occurred while generating the answer: {e}"

    return generated_answer, retrieved_documents


print_markdown("## Answers to Topic-Focused Questions (without metadata):")
for q in questions_topic:
    print_markdown(f"### Question: {q}")
    answer, snippets = simple_rag_pipeline(q)
    print_markdown(f"**Answer:** {answer}")
    print_markdown("**Retrieved Snippets:**")
    for i, snip in enumerate(snippets):
        print_markdown(f"- {snip}")
    print_markdown("---")

print_markdown("## Answers to Company-Focused Questions (without metadata):")
for q in questions_company:
    print_markdown(f"### Question: {q}")
    answer, snippets = simple_rag_pipeline(q)
    print_markdown(f"**Answer:** {answer}")
    print_markdown("**Retrieved Snippets:**")
    for i, snip in enumerate(snippets):
        print_markdown(f"- {snip}")
    print_markdown("---")

print_markdown("## Answers to Industry-Focused Questions (without metadata):")
for q in questions_industry:
    print_markdown(f"### Question: {q}")
    answer, snippets = simple_rag_pipeline(q)
    print_markdown(f"**Answer:** {answer}")
    print_markdown("**Retrieved Snippets:**")
    for i, snip in enumerate(snippets):
        print_markdown(f"- {snip}")
    print_markdown("---")

## Answers to Topic-Focused Questions (without metadata):

### Question: What are the major concerns expressed in financial news about inflation?

**Answer:** The major concerns expressed in financial news about inflation include persistent US inflation risks highlighted by the Federal Reserve, food inflation dampening hopes for a rate cut, and the impact of new tariffs causing grocery bills to rise. Additionally, there are worries about the uncertain macroeconomic outlook affecting earnings expectations and the overall economic slowdown.

**Retrieved Snippets:**

- Bitcoin price slips as Fed minutes flag US inflation risks : The Federal Reserve’s May policy meeting revealed mounting concern over persistent US inflation and the potential for economic slowdown.

...

- Is Schlumberger Limited (SLB) the Most Undervalued Energy Stock to Buy According to Hedge Funds? : We recently published a list of the 10 Most Undervalued Energy Stocks to Buy According to Hedge Funds. In this article, we are going to take a look at where Schlumberger Limited (NYSE:SLB) stands against other undervalued energy stocks. As of the close of May 2, 2025, the overall energy sector is undervalued by 13.1%, […]

- PG&E Corporation (PCG): Among the Best Energy Stocks to Buy Right Now : We recently published a list of the 13 Best Energy Stocks to Buy Right Now. In this article, we are going to take a look at where PG&E Corporation (NYSE:PCG) stands against other best energy stocks. The worldwide energy industry has recently been rattled by a combination of factors, including the trade war sparked by President […]

- Sempra Energy Rides on LNG Operations & Strategic Investments : SRE is expected to gain from planned investments and LNG projects. Yet, valuation concerns and wildfire events may impact its operations.

- Sector Update: Energy Stocks Rise Thursday Afternoon : Energy stocks rose Thursday afternoon with the NYSE Energy Sector Index up 0.1% and the Energy Selec

---

### 🔎 Note on the provided `FaissVectorStore.search`

The class supplied in the template has a defect in its **filtered** branch that is not
triggered above (the v1 pipeline passes `metadata_filter=None`), but that would silently
break any metadata filtering:

```python
D, I = temp_index.search(query_embedding, k)
indices = [filtered_indices[i] for i in I[0]]
# ...  D is never unwrapped to D[0] here
```

In the unfiltered branch the code does `D = D[0]`; in the filtered branch it does not, so
`zip(indices, D)` pairs the *first* index with the whole score row and returns a **single
document instead of k**. Passing `k` straight to a smaller sub-index also makes FAISS return
`-1` padding when fewer than `k` documents survive the filter. Both are corrected in
`search_with_metadata` in the RAG v2 section.

In [17]:
# =====================================================================
# Retrieval quality of RAG v1 - measured, not impressionistic
# =====================================================================
# Two cheap diagnostics computed directly from the retrieved sets (no API calls):
#   precision@k : share of retrieved snippets that mention the subject of the question
#   duplicates  : slots in the top-k consumed by a document already retrieved
import re

RELEVANCE_PATTERNS = {
    "inflation":               [r'inflat', r'\bcpi\b', r'tariff', r'\bfed\b|federal reserve',
                                r'rate cut', r'cost of living'],
    "investor sentiment":      [r'sentiment', r'bullish', r'bearish', r'price target',
                                r'\brating', r'analyst', r'outlook', r'investor'],
    "artificial intelligence": [r'\bai\b', r'artificial intelligence'],
    "Microsoft":               [r'microsoft', r'\bmsft\b'],
    "Amazon":                  [r'amazon', r'\bamzn\b', r'\baws\b'],
    "semiconductor":           [r'semiconductor', r'\bchip', r'foundry', r'nvidia', r'\bamd\b',
                                r'intel\b', r'broadcom', r'lam research', r'qualcomm', r'micron'],
    "retail":                  [r'retail', r'\bstore', r'consumer', r'shopper', r'merchandis',
                                r'e-?commerce', r'apparel'],
    "energy":                  [r'\benergy\b', r'\boil\b', r'\bgas\b', r'renewable', r'solar',
                                r'utilit', r'drill', r'\bpower\b', r'pipeline'],
}
# Most specific subject first, so "Microsoft ... AI" is scored on Microsoft.
SUBJECT_ORDER = ["Microsoft", "Amazon", "semiconductor", "retail", "energy",
                 "inflation", "investor sentiment", "artificial intelligence"]

def subject_of(question):
    return next(s for s in SUBJECT_ORDER if s.lower() in question.lower())

def retrieval_report(questions, k=K):
    rows = []
    for q in questions:
        subject = subject_of(q)
        patterns = RELEVANCE_PATTERNS[subject]
        docs = [doc for doc, _, _ in faiss_store.search(q, k=k)]

        on_topic = sum(1 for d in docs if any(re.search(p, d, re.I) for p in patterns))
        unique_docs = {d[:120].lower() for d in docs}
        # distinct articles that actually mention the subject
        distinct_on_topic = len({d[:120].lower() for d in docs
                                 if any(re.search(p, d, re.I) for p in patterns)})

        rows.append({
            'SUBJECT': subject,
            f'ON_TOPIC@{k}': on_topic,
            f'PRECISION@{k}': round(on_topic / len(docs), 2) if docs else 0,
            'DISTINCT_ON_TOPIC': distinct_on_topic,
            'UNIQUE_DOCS': len(unique_docs),
            'DUPLICATE_SLOTS': len(docs) - len(unique_docs),
        })
    return pd.DataFrame(rows)

v1_retrieval_report = retrieval_report(questions_topic + questions_company + questions_industry)
display(v1_retrieval_report)
n_q = len(v1_retrieval_report)
print(f"\nMean precision@{K}:", round(v1_retrieval_report[f'PRECISION@{K}'].mean(), 2))
print(f"Duplicate slots across all {n_q} queries:",
      int(v1_retrieval_report['DUPLICATE_SLOTS'].sum()), f"of {n_q * K}")

,SUBJECT,ON_TOPIC@25,PRECISION@25,DISTINCT_ON_TOPIC,UNIQUE_DOCS,DUPLICATE_SLOTS
0,inflation,10,0.40,7,21,4
1,investor sentiment,14,0.56,13,21,4
2,artificial intelligence,22,0.88,20,22,3
3,Microsoft,5,0.20,2,20,5
4,Amazon,5,0.20,5,17,8
5,semiconductor,11,0.44,10,24,1
6,retail,19,0.76,14,20,5
7,energy,20,0.80,17,22,3



Mean precision@25: 0.53
Duplicate slots across all 8 queries: 33 of 200


## Analysis & Questions - Section 1

### Analysis and Reflection on Retrieval and Generation Results
After running the RAG pipeline and obtaining answers along with their supporting news excerpts, take some time to carefully review both the generated responses and the retrieved contexts.

- **For each question, read the answer and then the corresponding news snippets used as context.**

- Reflect on the following points and document your observations:
1. **Relevance**
2. **Completeness**  
3. **Bias or Noise**
4. **Consistency**  
5. **Improvement Ideas**   

and answer the questions below:

#### **Question 1.** How well do the retrieved news snippets support the generated answer? Are the key facts or themes in the answer clearly grounded in the context?

**Grounding is high, but its strength varies sharply with question type, and the
retrieval diagnostics show why.**

> *Note on figures.* The retrieval metrics quoted below (precision@25, duplicate counts) are
> **deterministic** — they depend only on the embeddings and re-running reproduces them exactly.
> The generated answers are not; see Question 4.

For the *topic* and *industry* questions, claims trace back to specific snippets: the retail
answer reproduces the −13.7% six-month move in retail stocks against the S&P 500's −5.5% loss
and the pre-tariff import surge; the energy answer maps onto identifiable headlines (the
congressional bill that would repeal renewable subsidies, the oilfield-service names
SLB/HAL/BKR, Xcel Energy's call for a federal solution to wildfire litigation). These are also
the queries with the best retrieval: **precision@25 of 0.76 for retail and 0.80 for energy**.
Nothing materially unsupported appears in those answers, so the
`answer based ONLY on the provided context` instruction is being respected.

The *company* questions expose the weak link. Asked about Microsoft and AI, the model produced
a five-word refusal — the correct behaviour given its context. The diagnostic cell shows
**5 of 25 retrieved snippets mention Microsoft (precision@25 = 0.20), but they correspond to
only 2 distinct articles**: one AI-market overview, and a ServiceNow story ("ServiceNow
Regenerates On Swarm Of AI Deals With Amazon, Microsoft And More") that appears **four times**
in the same top-25. Neither is *about* Microsoft; in both, Microsoft is a passing mention. So
the failure was in **retrieval**, not generation: the question's embedding is dominated by
"artificial intelligence", and the entity carries little weight in a 384-dimensional MiniLM
vector over a corpus saturated with AI headlines. The diagnosis is confirmed later — restricting
retrieval to `TICKER == 'MSFT'` yields a grounded answer from 10 genuine Microsoft articles using
the same model and prompt.

Two caveats. Answers state themes as established facts when each rests on one or two headlines
out of 4,871 articles: faithful to the context, but the context is not necessarily
representative of the corpus. And grounding is asserted rather than shown — the v1 prompt
requests no citations, so the reader cannot check which snippet supports which claim without
reading all 25.

#### **Question 2.** Does the answer fully address the question, or does it leave important aspects out? Consider if the retrieved context provided enough information to generate a thorough response.

**Partially. Coverage is adequate for broad questions and clearly incomplete for
entity-specific ones — and answer length makes the gap measurable.**

In the main v1 loop of this run, answers range from **5 to 200 words (median ≈ 86)**. Exact
counts shift between runs (Question 4), but the shape of the distribution does not:

- *Inflation* (55 words): persistent US inflation risk, food inflation delaying rate cuts,
  tariff-driven grocery bills, macro uncertainty. A reasonable set of themes, thin on magnitude,
  direction and timing. Retrieval precision@25 here is only 0.40, so more than half the context
  contributed nothing.
- *Microsoft* (5 words): not addressed at all — see Question 1. This is the one answer that is
  identical across both v1 executions, because a refusal has nothing to vary.
- *Amazon* (128 words): lists three headlines, and in none of them is Amazon the subject — UPS
  selling Ware2Go to Stord *in order to compete with* Amazon; an Nvidia/AI-data-centre roundup
  that mentions AWS leases; and a Woodward automation note that the answer itself concedes
  "primarily focuses on Woodward". Precision@25 = 0.20. Notably, the AMZN-tagged article
  "Amazon's AI Roadmap With AWS CEO Garman" **was** retrieved but did not make it into the
  answer — so this is not only a retrieval failure but a ranking one: the genuine Amazon story
  sat below three peripheral mentions.
- *Semiconductors* (200 words) and *energy* (168 words): the most complete answers, and among
  the better-retrieved sets (precision 0.44 and 0.80).

Two structural limits cap depth. Only `TITLE + SUMMARY` is embedded and passed to the model, so
each "document" is two or three sentences — even at k = 25 the model receives headlines rather
than reporting. And because the retrieved set is never counted or aggregated, the model cannot
say whether a theme appears in 2 articles or 200, so it cannot distinguish a dominant narrative
from an isolated one.

#### **Question 3.** Are there any irrelevant or misleading snippets retrieved that may have influenced the answer? How might this affect the quality of the output?

**Yes — and the diagnostic cell quantifies how much of the context is wasted.**

Mean precision@25 across the eight questions is **0.53**, i.e. roughly half of every context
window is off-topic, and **33 of 200 retrieved slots (16.5%) are duplicates of a document
already present**. Four distinct failure modes:

1. **Entity drift.** The Microsoft query returns 25 snippets of which only 2 distinct articles
   name Microsoft at all, and neither is about it (precision@25 = 0.20). With a context full of
   *other* companies' AI stories, a less conservative model would generalise "the AI narrative"
   onto Microsoft — plausible-sounding and unsupported. The model refused, which is the safe
   outcome, but the exposure is structural.
2. **Sector drift.** The v1 energy answer includes a paragraph on the *industrial* sector's
   six-month stock decline — neither about energy nor about the risks asked about. It entered
   on lexical similarity and was dutifully summarised as an energy risk.
3. **Attribution risk.** The Amazon answer is built on a headline about a *rival* (UPS/Stord)
   expanding to compete with Amazon. Summarised loosely that becomes "Amazon faces competition
   in logistics" — defensible — but one step from attributing Stord's expansion to Amazon.
4. **Redundancy — measured.** The Amazon query wastes 8 of 25 slots on duplicates (17 unique
   documents); the Microsoft query wastes 5, four of them on the *same* ServiceNow article. A
   syndicated story therefore acts as implicit vote-weighting: repetition makes it look more
   important to the model than it is.

The combined effect is dilution. At precision 0.2, four out of five snippets argue for
something other than what was asked, and the model must hedge, refuse, or drift. Precision
matters more than recall here, which argues for a smaller k *after* filtering rather than a
large k before it.

#### **Question 4.**  Do the news snippets show consistent information, or are there conflicting viewpoints? How does the LLM handle potential contradictions in the context?

**The snippets are frequently in tension, the model concatenates rather than
adjudicates, and the pipeline is less deterministic than it looks.**

The clearest case is investor sentiment: the retrieved set contains bullish price targets with
large implied upside, commentary on analysts' structural reluctance to issue sell ratings, and
outright bearish forecasts. The answer presents all three — "overwhelmingly bullish... however...
bearish forecasts" — honest but unresolved. The semiconductor answer does the same, listing ON
Semiconductor's Q1 net loss alongside its 30% monthly price surge as parallel themes rather than
as a contradiction worth explaining.

So the model's strategy is *coordination*: it stacks opposing claims with connectives
("however", "additionally") instead of weighing them. It never asks which source is more
credible, which is more recent, or how many documents support each side. That follows directly
from how the context is built — `PUBLICATION_DATE` and `PROVIDER` exist in the store but are
stripped before the prompt, so a May 19 speculative listicle and a May 29 earnings report are
indistinguishable to the model.

**Non-determinism, measured.** The v1 pipeline is executed twice on the same questions in this
notebook (the main loop and the comparison cell), and **6 of the 8 answers differ between the two
executions**. Most differences are cosmetic, but not all: on the energy question one execution
says the subsidy-repeal bill is "advancing in Congress" while the other states it was "passed by
the U.S. House of Representatives" — a factual claim about legislative status that changed
between runs from the same corpus, the same k and `temperature=0.1`. Answer lengths move too
(the semiconductor answer was 200 words in one execution and 222 in the other). Only the
retrieval layer is stable: precision@25 and duplicate counts reproduce exactly. Low temperature
therefore reduces variance without eliminating it, so a single output should never be treated as
the pipeline's answer; evaluation should average over several runs.

#### **Question 5.**  Based on your observations, suggest ways the retrieval or generation process could be improved (e.g., better filtering, adjusting `k`, refining prompt design).

**Retrieval fixes (highest impact first)**

1. **Metadata pre-filtering.** Detect a company, sector or industry in the question and restrict
   the search space before computing similarity. RAG v2 below implements this and the executed
   results confirm it is the decisive fix for entity questions: the Microsoft query goes from a
   five-word refusal to a ~160-word answer built on 10 genuine `MSFT` articles, and the Amazon
   query from three peripheral headlines to 10 `AMZN` articles.
   ⚠️ Filtering must be conservative. The first version of these filters introduced two defects,
   documented in Section 2: a generic token ("financial") matching the industry *Financial Data &
   Stock Exchanges*, and constraints combined with AND, which drove the Amazon query to zero
   documents. Both are corrected, and the naive version is kept as an ablation arm in the
   comparison cell — it still triggers 1 fallback where the corrected version triggers 0.
2. **Put the entity in the embedded text.** `EMBEDDED_TEXT` is `TITLE + SUMMARY`; prepending
   ticker, company name, sector and industry
   (`"MSFT | Microsoft Corporation | Technology | Software - Infrastructure : <title> : <summary>"`)
   would put the entity inside the vector instead of leaving it implied. Unlike filtering it
   degrades gracefully — it biases retrieval rather than excluding documents outright.
3. **Hybrid retrieval.** Combine dense similarity with lexical BM25. Exact-token matching is what
   dense embeddings are worst at, and entity names are exact tokens.
4. **Deduplicate** on URL and normalised title. Measured value: 33 of 200 slots recovered across
   the eight questions, 8 of them on the Amazon query alone.
5. **Re-rank, then shrink k.** Retrieve 50 candidates, re-rank with a cross-encoder
   (e.g. `ms-marco-MiniLM-L-6-v2`), keep the best 8. MMR is a cheaper alternative for diversity.
   At mean precision 0.53, k = 25 is buying noise, not coverage. The Amazon case shows re-ranking
   matters independently of filtering: the right document was retrieved but ranked below three
   peripheral ones.
6. **Chunk full article text** instead of title + summary, so the context carries reporting
   rather than headlines.

**Generation fixes**

7. **Pass metadata into the prompt** and number the documents. Measured effect: citations go from
   0.0 per answer in v1 to ≈ 6 in v2.1.
8. **Require explicit citations and explicit abstention** ("if no document names the company in
   the question, say so").
9. **Ask the model to flag disagreement** rather than stacking it.
10. **Weight recency** by exposing `PUBLICATION_DATE` and instructing the model to prefer the
    most recent evidence when sources conflict.

**Evaluation**

11. The `retrieval_report` cell above is a first step: precision@k and duplicate counts turn
    "the retrieval is noisy" into a number that can be tracked, and unlike the answers themselves
    those numbers are reproducible. Extend it with a labelled set (question → expected supporting
    articles) and, given that 6 of 8 answers changed between two executions of the same pipeline
    (Question 4), average over 3–5 runs before concluding that a change helped.

## 🧠 Retrieval-Augmented Generation (RAG) v2: Adding Financial Metadata to Improve Generation

👉 **Instructions**:

In this part of the assignment, you’ll enhance your Retrieval-Augmented Generation (RAG) pipeline by incorporating *financial metadata* to provide more contextually rich answers.

Your goal is to evaluate whether metadata such as **company name**, **sector**, and **industry** helps the LLM generate **more accurate and grounded answers** to financial questions.

---

### ✅ What your updated pipeline should do:

- Retrieve relevant financial news articles using semantic similarity with FAISS.
- Enrich each retrieved document with financial metadata:
  - Ticker symbol
  - Full company name
  - Sector (e.g., Technology, Energy)
  - Industry (e.g., Semiconductors, Retail)
- Construct prompts that include both:
  - Retrieved news text
  - Associated metadata
- Send the prompt to the OpenAI model to generate an informed response.
- Return:
  - The final answer
  - The exact set of contextual documents used to produce that answer

---

### 🧪 Evaluation and Comparison:

You will test your improved RAG pipeline on the same three types of questions provided earlier:
- **Topic-focused** (e.g., inflation, interest rates)
- **Company-focused** (e.g., questions about Tesla, Nvidia)
- **Industry-focused** (e.g., semiconductors, utilities)


In [18]:
# =====================================================================
# RAG v2 - Step 1: build the metadata layer
# =====================================================================
import re

# Ticker -> company / sector / industry lookup (from the yfinance step)
ticker_meta = (
    df_meta.set_index('TICKER')[['COMPANY_NAME', 'SECTOR', 'INDUSTRY']]
           .to_dict('index')
)

def enrich_metadata(meta):
    """Add COMPANY_NAME / SECTOR / INDUSTRY to a base metadata record."""
    extra = ticker_meta.get(meta['TICKER'], {})
    return {
        **meta,
        'COMPANY_NAME': extra.get('COMPANY_NAME', 'N/A'),
        'SECTOR':       extra.get('SECTOR', 'N/A'),
        'INDUSTRY':     extra.get('INDUSTRY', 'N/A'),
    }

# ---- Vocabularies used to detect entities mentioned in a question ----
SECTORS    = sorted({s for s in df_meta['SECTOR'].unique()   if s not in ('N/A', None)})
INDUSTRIES = sorted({i for i in df_meta['INDUSTRY'].unique() if i not in ('N/A', None)})

_LEGAL_SUFFIXES = r'\b(inc|corp|corporation|company|co|plc|ltd|limited|holdings|holding|group|incorporated|the|class|international|technologies)\b'

def _short_name(name):
    """'Microsoft Corporation' -> 'microsoft'  (a matchable brand token)"""
    n = name.lower().replace(',', ' ').replace('.', ' ').replace('&', ' ')
    n = re.sub(_LEGAL_SUFFIXES, ' ', n)
    return ' '.join(n.split())

# Brand tokens that are ordinary English words would fire on unrelated questions
# ("target price", "the gap between..."), so they are excluded from name matching.
AMBIGUOUS_NAMES = {'target', 'ball', 'block', 'match', 'news', 'first', 'global'}

SHORT_NAME_TO_TICKER = {}
for row in df_meta.itertuples():
    if not row.COMPANY_NAME or row.COMPANY_NAME == 'N/A':
        continue                       # tickers without a name cannot be matched by name
    full = _short_name(row.COMPANY_NAME)           # 'microsoft', 'amazon com'
    first = full.split()[0] if full else ''         # 'amazon'  <- brand alias
    for key in {full, first}:
        if len(key) >= 4 and key not in AMBIGUOUS_NAMES:
            SHORT_NAME_TO_TICKER.setdefault(key, row.TICKER)

print(f"Company names available for matching: {len(SHORT_NAME_TO_TICKER)} "
      f"({(df_meta['COMPANY_NAME'] == 'N/A').sum()} tickers unmatchable by name)")

# ---------------------------------------------------------------------
# Tokens that appear inside industry names but are far too generic to be
# evidence that a question is *about* that industry. Without this list the
# word "financial" in "financial news" matches the industry
# 'Financial Data & Stock Exchanges' and silently narrows almost every query.
# ---------------------------------------------------------------------
GENERIC_INDUSTRY_TOKENS = {
    'financial', 'finance', 'stock', 'stocks', 'market', 'markets', 'capital',
    'general', 'specialty', 'diversified', 'services', 'products', 'equipment',
    'holding', 'holdings', 'industry', 'industries', 'systems', 'solutions',
    'international', 'global', 'other',
}


def _raw_matches(query):
    """Every ticker / sector / industry the question could plausibly refer to."""
    q = query.lower()

    tickers = {t for name, t in SHORT_NAME_TO_TICKER.items()
               if re.search(rf'\b{re.escape(name)}\b', q)}

    sectors = {s for s in SECTORS if re.search(rf'\b{re.escape(s.lower())}\b', q)}

    industries = set()
    for ind in INDUSTRIES:
        tokens = [w for w in re.split(r'[^a-z]+', ind.lower())
                  if len(w) >= 5 and w not in GENERIC_INDUSTRY_TOKENS]
        for w in tokens:
            stem = w[:-1] if w.endswith('s') else w    # 'semiconductors' -> 'semiconductor'
            if re.search(rf'\b{re.escape(stem)}s?\b', q):
                industries.add(ind)
                break

    return tickers, sectors, industries


def detect_filters(query):
    """
    Infer retrieval constraints from a natural-language question.

    PRECEDENCE, not intersection: the most specific signal wins. A question that
    names a company must not also be constrained by an industry guessed from the
    same sentence - intersecting the two is what returned 0 documents for the
    Amazon query in the first version of this notebook.
    """
    tickers, sectors, industries = _raw_matches(query)

    if tickers:
        return {'TICKER': tickers, 'SECTOR': set(), 'INDUSTRY': set()}
    if industries:
        return {'TICKER': set(), 'SECTOR': set(), 'INDUSTRY': industries}
    return {'TICKER': set(), 'SECTOR': sectors, 'INDUSTRY': set()}


def detect_filters_naive(query):
    """
    First version, kept only as the ablation baseline in the comparison below:
    no generic-token blocklist, and all constraints combined with AND.
    """
    q = query.lower()
    tickers = {t for name, t in SHORT_NAME_TO_TICKER.items()
               if re.search(rf'\b{re.escape(name)}\b', q)}
    sectors = {s for s in SECTORS if re.search(rf'\b{re.escape(s.lower())}\b', q)}
    industries = set()
    _STOP = {'other', 'general', 'specialty', 'diversified', 'services', 'products', 'equipment'}
    for ind in INDUSTRIES:
        for w in [w for w in re.split(r'[^a-z]+', ind.lower()) if len(w) >= 5 and w not in _STOP]:
            stem = w[:-1] if w.endswith('s') else w
            if re.search(rf'\b{re.escape(stem)}s?\b', q):
                industries.add(ind)
                break
    return {'TICKER': tickers, 'SECTOR': sectors, 'INDUSTRY': industries}


def build_metadata_filter(constraints):
    """Turn detected constraints into a predicate for the vector store."""
    if not any(constraints.values()):
        return None

    def _filter(meta):
        full = enrich_metadata(meta)
        if constraints['TICKER'] and full['TICKER'] not in constraints['TICKER']:
            return False
        if constraints['SECTOR'] and full['SECTOR'] not in constraints['SECTOR']:
            return False
        if constraints['INDUSTRY'] and full['INDUSTRY'] not in constraints['INDUSTRY']:
            return False
        return True

    return _filter


# ---- Side-by-side check: naive filters vs corrected filters -------------
print(f"\n{'QUESTION':<58} {'NAIVE (v2.0)':<46} CORRECTED (v2.1)")
for q in questions_topic + questions_company + questions_industry:
    naive = {k_: sorted(v) for k_, v in detect_filters_naive(q).items() if v}
    fixed = {k_: sorted(v) for k_, v in detect_filters(q).items() if v}
    print(f"{q[:56]:<58} {str(naive)[:44]:<46} {str(fixed)[:60]}")

Company names available for matching: 656 (11 tickers unmatchable by name)

QUESTION                                                   NAIVE (v2.0)                                   CORRECTED (v2.1)
What are the major concerns expressed in financial news    {'INDUSTRY': ['Financial Data & Stock Exchan   {}
How is investor sentiment described in recent financial    {'INDUSTRY': ['Financial Data & Stock Exchan   {}
What role is artificial intelligence playing in recent f   {}                                             {}
How is Microsoft being portrayed in news stories about a   {'TICKER': ['MSFT']}                           {'TICKER': ['MSFT']}
What financial news headlines connect Amazon with automa   {'TICKER': ['AMZN'], 'INDUSTRY': ['Financial   {'TICKER': ['AMZN']}
What are the main themes emerging in financial news abou   {'INDUSTRY': ['Financial Data & Stock Exchan   {'INDUSTRY': ['Semiconductor Equipment & Materials', 'Semico
What trends are being reported in the retail industry

In [19]:
# =====================================================================
# RAG v2 - Step 2: a corrected, metadata-aware search function
# =====================================================================
# NOTE: FaissVectorStore.search has a bug in its filtered branch - it forgets
# to unwrap D (D[0]), so `zip(indices, D)` yields a single pair and only one
# document comes back. It also passes k straight to a smaller sub-index, which
# makes FAISS return -1 padding when the filtered set has fewer than k items.
# The function below fixes both and adds de-duplication.

def search_with_metadata(store, query, k=K, metadata_filter=None, dedup=True):
    """Return [(document, enriched_metadata, similarity), ...] sorted by similarity."""
    q_emb = store.model.encode([query])
    q_emb = (q_emb / np.linalg.norm(q_emb)).astype('float32')

    if metadata_filter is not None:
        keep = [i for i, m in enumerate(store.metadata) if metadata_filter(m)]
        if not keep:
            return []
        sub = store.embeddings[keep].astype('float32')
        tmp = faiss.IndexFlatIP(sub.shape[1])
        tmp.add(sub)
        D, I = tmp.search(q_emb, min(k, len(keep)))
        pairs = [(keep[i], float(s)) for i, s in zip(I[0], D[0]) if i != -1]
    else:
        D, I = store.index.search(q_emb, k)
        pairs = [(int(i), float(s)) for i, s in zip(I[0], D[0]) if i != -1]

    results, seen = [], set()
    for idx, sim in pairs:
        doc = store.documents[idx]
        if dedup:
            key = doc[:120].lower()      # collapses syndicated duplicates
            if key in seen:
                continue
            seen.add(key)
        results.append((doc, enrich_metadata(store.metadata[idx]), sim))
    return results


def format_context(results):
    """Render retrieved documents as a numbered, metadata-annotated context block."""
    blocks = []
    for n, (doc, meta, sim) in enumerate(results, start=1):
        header = (f"[{n}] {meta['TICKER']} - {meta['COMPANY_NAME']} | "
                  f"Sector: {meta['SECTOR']} | Industry: {meta['INDUSTRY']} | "
                  f"Date: {meta['PUBLICATION_DATE']} | Source: {meta['PROVIDER']} "
                  f"| similarity: {sim:.3f}")
        blocks.append(f"{header}\n{doc}")
    return "\n\n".join(blocks)

In [20]:
# =====================================================================
# RAG v2 - Step 3: the metadata-aware pipeline
# =====================================================================
SYSTEM_PROMPT_V2 = """You are a financial research assistant analysing S&P 500 news.

Rules:
1. Answer ONLY from the numbered documents in the context. Never use outside knowledge.
2. Cite the document number(s) in square brackets after every claim, e.g. [3].
3. Use the metadata: refer to companies by name and ticker, and group findings by sector
   or industry when that makes the answer clearer.
4. Prefer the most recent documents when sources disagree, and say explicitly when
   sources conflict.
5. If no document explicitly mentions the company, sector or topic asked about, say so
   plainly instead of generalising from unrelated documents.
Keep the answer concise (roughly 150-250 words)."""


def rag_pipeline_v2(query, k=K, filter_fn=detect_filters, verbose=False):
    """
    Retrieve with metadata filtering, generate with a metadata-enriched context.

    filter_fn lets the comparison cell run the same pipeline with the naive
    (v2.0) and corrected (v2.1) filter strategies.
    """
    empty = {'TICKER': set(), 'SECTOR': set(), 'INDUSTRY': set()}
    constraints = filter_fn(query) if filter_fn else empty
    meta_filter = build_metadata_filter(constraints)

    results = search_with_metadata(faiss_store, query, k=k, metadata_filter=meta_filter)
    fell_back = False

    # Fallback: if the filter is too strict (or matched nothing), search the full corpus.
    if len(results) < 3 and meta_filter is not None:
        if verbose:
            print(f"  -> filter too narrow ({len(results)} docs), falling back to full corpus")
        results = search_with_metadata(faiss_store, query, k=k, metadata_filter=None)
        constraints, fell_back = empty, True

    if not results:
        return "No relevant documents found to answer your question.", [], constraints, fell_back

    context = format_context(results)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V2},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"},
    ]

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.1,
        )
        answer = response.choices[0].message.content
    except Exception as e:
        answer = f"An error occurred while generating the answer: {e}"

    return answer, results, constraints, fell_back


def show_v2(query):
    answer, results, constraints, fell_back = rag_pipeline_v2(query, verbose=True)
    applied = {k_: sorted(v) for k_, v in constraints.items() if v}
    label = applied if applied else ('none - fell back to full corpus' if fell_back else 'none (full corpus)')
    print_markdown(f"### Question: {query}")
    print_markdown(f"*Filters applied:* `{label}` - *{len(results)} documents retrieved*")
    print_markdown(f"**Answer:** {answer}")
    print_markdown("**Retrieved snippets (with metadata):**")
    for n, (doc, meta, sim) in enumerate(results, start=1):
        print_markdown(
            f"- **[{n}] {meta['TICKER']} - {meta['COMPANY_NAME']}** "
            f"({meta['SECTOR']} / {meta['INDUSTRY']}, {meta['PUBLICATION_DATE']}, "
            f"sim {sim:.3f})<br>{doc}"
        )
    print_markdown("---")
    return answer, results

In [21]:
# =====================================================================
# RAG v2 - Step 4: run on the three families of questions
# =====================================================================
answers_v2 = {}

print_markdown("## Answers to Topic-Focused Questions (WITH metadata):")
for q in questions_topic:
    answers_v2[q] = show_v2(q)[0]

print_markdown("## Answers to Company-Focused Questions (WITH metadata):")
for q in questions_company:
    answers_v2[q] = show_v2(q)[0]

print_markdown("## Answers to Industry-Focused Questions (WITH metadata):")
for q in questions_industry:
    answers_v2[q] = show_v2(q)[0]

## Answers to Topic-Focused Questions (WITH metadata):

### Question: What are the major concerns expressed in financial news about inflation?

*Filters applied:* `none (full corpus)` - *21 documents retrieved*

**Answer:** Recent financial news highlights several major concerns regarding inflation:

1. **Persistent Inflation Risks**: The Federal Reserve's May policy meeting revealed growing worries about persistent inflation and its potential to lead to an economic slowdown. This concern is underscored by the Fed's commentary, which suggests that inflation remains a significant issue for policymakers [1].

2. **Impact on Consumer Prices**: Inflation is affecting everyday consumer goods, as seen in reports about rising grocery prices. For instance, Dollar Tree (DLTR) noted that creeping inflation and new tariffs have significantly increased weekly grocery bills, impacting consumer spending [3].

3. **Economic Outlook**: Broader economic uncertainties are overshadowing corporate earnings, with analysts indicating that the grim economic outlook is taking precedence over better-than-expected earnings results. This sentiment reflects a cautious approach among investors as they navigate the implications of inflation and tariffs [16].

4. **Geopolitical Tensions**: Rising geopolitical tensions are also contributing to inflation concerns, particularly in the context of hard assets like gold. Investors are increasingly seeking stability in these assets amid fears of fiscal sustainability and currency devaluation [11].

Overall, the combination of persistent inflation, rising consumer prices, economic uncertainty, and geopolitical factors creates a complex landscape for investors and consumers alike.

**Retrieved snippets (with metadata):**

...

- **[21] EXE - Expand Energy Corporation** (Energy / Oil & Gas E&P, 2025-05-23, sim 0.386)<br>Bernstein Initiates Coverage of Expand Energy (EXE) : It was recently reported that analysts at Bernstein have initiated coverage of Expand Energy Corporation (NASDAQ:EXE). Let’s shed some light on the development. Formed in 2024 by the merger of Chesapeake Energy Corporation and Southwestern Energy Company, Expand Energy Corporation (NASDAQ:EXE) is the largest natural gas producer in America. The company is focused on responsibly […]

- **[22] EXE - Expand Energy Corporation** (Energy / Oil & Gas E&P, 2025-05-20, sim 0.385)<br>Are Oils-Energy Stocks Lagging  Expand Energy Corporation (EXE) This Year? : Here is how Expand Energy (EXE) and CSLM Acquisition Corp. (SPWR) have performed compared to their sector so far this year.

- **[23] VLO - Valero Energy Corporation** (Energy / Oil & Gas Refining & Marketing, 2025-05-14, sim 0.382)<br>Valero Energy Corporation (VLO): Among the Stocks Analysts Are Upgrading Today : We recently compiled a list of the 10 Stocks Analysts Are Upgrading Today. In this article, we are going to take a look at where Valero Energy Corporation (NYSE:VLO) stands against the other stocks analysts are upgrading today. The easing of the US-China trade war is the catalyst driving equity markets higher after weeks of heightened […]

- **[24] WMB - The Williams Companies, Inc.** (Energy / Oil & Gas Midstream, 2025-05-30, sim 0.381)<br>Data Center & Natural Gas Link Grows: Will WMB, ENB, KMI Stocks Gain? : WMB, ENB and KMI aim to tap rising AI data center energy needs as natural gas demand drives major infrastructure investments and growth prospects.

---

In [22]:
# =====================================================================
# RAG v2 - Step 5: three-way comparison and quantitative evaluation
# =====================================================================
# Arm A: no metadata            (v1 - plain semantic search, plain prompt)
# Arm B: metadata, naive filters (v2.0 - the version whose defects are documented)
# Arm C: metadata, fixed filters (v2.1 - blocklist + precedence)
all_questions = questions_topic + questions_company + questions_industry

answers_v1, answers_v20, meta_v20, meta_v21 = {}, {}, {}, {}

for q in all_questions:
    answers_v1[q] = simple_rag_pipeline(q)[0]

    ans20, res20, cons20, fb20 = rag_pipeline_v2(q, filter_fn=detect_filters_naive)
    answers_v20[q] = ans20
    meta_v20[q] = (len(res20), fb20, {k_: sorted(v) for k_, v in detect_filters_naive(q).items() if v})

    _, res21, _, fb21 = rag_pipeline_v2(q, filter_fn=detect_filters)
    meta_v21[q] = (len(res21), fb21, {k_: sorted(v) for k_, v in detect_filters(q).items() if v})

# ---- Qualitative side-by-side -----------------------------------------
for q in all_questions:
    print_markdown(f"### {q}")
    print_markdown(f"**WITHOUT metadata (v1):**\n\n{answers_v1[q]}")
    print_markdown(f"**WITH metadata (v2.1):**\n\n{answers_v2.get(q, '(run Step 4 first)')}")
    print_markdown("---")

# ---- Quantitative evaluation table ------------------------------------
def n_citations(text):
    return len(re.findall(r'\[\d+\]', text or ''))

evaluation = pd.DataFrame({
    'SUBJECT':        [subject_of(q) for q in all_questions],
    'WORDS_V1':       [len(answers_v1[q].split()) for q in all_questions],
    'WORDS_V21':      [len(answers_v2.get(q, '').split()) for q in all_questions],
    'CITES_V1':       [n_citations(answers_v1[q]) for q in all_questions],
    'CITES_V21':      [n_citations(answers_v2.get(q, '')) for q in all_questions],
    'DOCS_V20':       [meta_v20[q][0] for q in all_questions],
    'DOCS_V21':       [meta_v21[q][0] for q in all_questions],
    'FELLBACK_V20':   [meta_v20[q][1] for q in all_questions],
    'FELLBACK_V21':   [meta_v21[q][1] for q in all_questions],
    'FILTER_V21':     [str(meta_v21[q][2]) if meta_v21[q][2] else 'none' for q in all_questions],
})
display(evaluation)

print("Mean words   : v1 =", round(evaluation['WORDS_V1'].mean(), 1),
      "| v2.1 =", round(evaluation['WORDS_V21'].mean(), 1))
print("Mean citations: v1 =", round(evaluation['CITES_V1'].mean(), 1),
      "| v2.1 =", round(evaluation['CITES_V21'].mean(), 1))
print("Fallbacks triggered: naive v2.0 =", int(evaluation['FELLBACK_V20'].sum()),
      "| corrected v2.1 =", int(evaluation['FELLBACK_V21'].sum()))

### What are the major concerns expressed in financial news about inflation?

**WITHOUT metadata (v1):**

The major concerns expressed in financial news about inflation include persistent US inflation risks highlighted by the Federal Reserve, food inflation dampening hopes for a rate cut, and the impact of new tariffs causing grocery bills to rise. Additionally, there are worries about the uncertain macroeconomic outlook and its effect on earnings expectations, as well as rising geopolitical tensions and skepticism about long-term fiscal discipline leading investors to seek stability in hard assets.

**WITH metadata (v2.1):**

Recent financial news highlights several major concerns regarding inflation:

1. **Persistent Inflation Risks**: The Federal Reserve's May policy meeting revealed growing worries about persistent inflation and its potential to lead to an economic slowdown. This concern is underscored by the Fed's commentary, which suggests that inflation remains a significant issue for policymakers [1].

2. **Impact on Consumer Prices**: Inflation is affecting everyday consumer goods, as seen in reports about rising grocery prices. For instance, Dollar Tree (DLTR) noted that creeping inflation and new tariffs have significantly increased weekly grocery bills, impacting consumer spending [3].

3. **Economic Outlook**: Broader economic uncertainties are overshadowing corporate earnings, with analysts indicating that the grim economic outlook is taking precedence over better-than-expected earnings results. This sentiment reflects a cautious approach among investors as they navigate the implications of inflation and tariffs [16].

4. **Geopolitical Tensions**: Rising geopolitical tensions are also contributing to inflation concerns, particularly in the context of hard assets like gold. Investors are increasingly seeking stability in these assets amid fears of fiscal sustainability and currency devaluation [11].

Overall, the combination of persistent inflation, rising consumer prices, economic uncertainty, and geopolitical factors creates a complex landscape for investors and consumers alike.

---

### How is investor sentiment described in recent financial headlines?

...

**WITHOUT metadata (v1):**

Recent news about the energy industry discusses several risks and challenges, including:

1. The potential repeal of important subsidies for the renewable energy industry due to a bill passed by the U.S. House of Representatives, which could make renewable projects uneconomical.
2. Oilfield service companies like SLB, HAL, and BKR facing a tough future due to sliding oil prices, rising tariffs, and shrinking drilling budgets.
3. The industrial sector experiencing a downturn, with significant declines in stock performance over the past six months, indicating a potential prolonged economic downturn.
4. Concerns about wildfire risks and litigation impacting companies like Xcel Energy and Edison International.
5. Valuation concerns and operational impacts related to wildfire events for companies like Sempra Energy. 

These factors contribute to a challenging environment for various segments within the energy industry.

**WITH metadata (v2.1):**

Recent news highlights several risks and challenges facing the energy industry, particularly within the oil and gas sector. 

1. **Falling Oil Prices**: A significant concern is the recent decline in oil prices, which has pressured producers to cut back on drilling activities and rethink their budgets. This situation has been exacerbated by increased output from OPEC+ and a global tariff war, leading to a challenging environment for oilfield service companies like Halliburton (HAL), Schlumberger (SLB), and Baker Hughes (BKR) [1][20].

2. **Rising Tariffs**: The imposition of tariffs is creating additional strain on the industry, impacting profitability and operational budgets for companies involved in oil and gas equipment and services [1][2].

3. **Investment in Technology**: While companies like Baker Hughes are expanding their technological ventures, the overall market remains cautious due to the volatile pricing environment and the need for significant capital investment in new technologies [18].

4. **Regulatory Challenges**: The U.S. government's recent decision to cancel awards for clean energy projects, including those at ExxonMobil's facilities, reflects a shifting regulatory landscape that could hinder growth in renewable energy initiatives [10].

These factors collectively indicate a turbulent period ahead for the energy sector, particularly for companies heavily reliant on oil prices and traditional drilling activities.

---

,SUBJECT,WORDS_V1,WORDS_V21,CITES_V1,CITES_V21,DOCS_V20,DOCS_V21,FELLBACK_V20,FELLBACK_V21,FILTER_V21
0,inflation,73,198,0,4,24,21,False,False,none
1,investor sentiment,65,188,0,8,23,21,False,False,none
2,artificial intelligence,87,157,0,3,22,22,False,False,none
3,Microsoft,5,161,0,4,10,10,False,False,{'TICKER': ['MSFT']}
4,Amazon,99,136,0,3,17,10,True,False,{'TICKER': ['AMZN']}
5,semiconductor,222,234,0,8,23,23,False,False,{'INDUSTRY': ['Semiconductor Equipment & Mater...
6,retail,77,216,0,12,25,25,False,False,"{'INDUSTRY': ['Apparel Retail', 'Home Improvem..."
7,energy,133,204,0,6,24,24,False,False,{'SECTOR': ['Energy']}


Mean words   : v1 = 95.1 | v2.1 = 186.8
Mean citations: v1 = 0.0 | v2.1 = 6.0
Fallbacks triggered: naive v2.0 = 1 | corrected v2.1 = 0


## Analysis & Questions - Section 2

### Instructions: Evaluate Answers With and Without Metadata

For each question, compare the two answers provided:
- One generated **without** metadata
- One generated **with** metadata

---

### Steps:

1. Use the following evaluation criteria:
   - Clarity
   - Detail & Depth
   - Use of Context
   - Accuracy & Grounding
   - Relevance
   - Narrrative Flow

2. For each criterion, write brief notes comparing how the answer **without metadata** performs versus the answer **with metadata**.

3. Summarize your evaluation in a markdown table with the following columns:

| Criteria       | WITHOUT METADATA            | WITH METADATA             |
|----------------|----------------------------|--------------------------|
| Clarity        | [Your brief note here]     | [Your brief note here]   |
| Detail & Depth         | [Your brief note here]     | [Your brief note here]   |
| Use of Context        | [Your brief note here]     | [Your brief note here]   |
| Accuracy & Grounding       | [Your brief note here]     | [Your brief note here]   |
| Relevance      | [Your brief note here]     | [Your brief note here]   |
| Narrative Flow      | [Your brief note here]     | [Your brief note here]   |

---

**Note:** Keep comments short and clear for easy comparison.



### Evaluation: answers without metadata vs. answers with metadata

All figures below come from the `evaluation` and `v1_retrieval_report` tables produced by the
cells above. Retrieval metrics (precision, duplicates, documents retrieved, fallbacks) are
deterministic and reproduce on re-run; answer lengths and citation counts are generated by the
model and move by roughly ±10% between runs, which is itself the finding of Section 1,
Question 4.

**Headline numbers**

| Metric | v1 (no metadata) | v2.1 (metadata) |
|---|---|---|
| Mean answer length | ≈ 95 words | ≈ 187 words |
| Mean citations per answer | 0.0 | ≈ 6.0 |
| Fallbacks to unfiltered corpus | n/a | 0 (naive v2.0 filters: 1) |
| Shortest answer | 5 words (Microsoft) | 136 words (Amazon) |

**Criteria**

| Criteria | WITHOUT METADATA | WITH METADATA |
|---|---|---|
| **Clarity** | Bare tickers; reader must know that JKHY is Jack Henry. | Ticker + full name + sector on every claim; answers grouped by sector. |
| **Detail & Depth** | 5–222 words, median ≈ 85; retail answer is one generic paragraph. | 136–234 words; retail answer names BBY, TJX, LOW, HD, ULTA, ROST, DASH and EBAY across four trends. |
| **Use of Context** | Date, ticker and provider discarded before the prompt; 33 of 200 slots lost to duplicates. | Full metadata reaches the model; duplicates removed; filtering spends the k budget on relevant documents. |
| **Accuracy & Grounding** | 0 citations in all 8 answers; imports an Industrials snippet as an energy risk. | 3–12 citations per answer; ticker filtering guarantees the retrieved set belongs to the company asked about. |
| **Relevance** | Precision@25 averages 0.53; collapses to 0.20 on both entity questions. | Sharply better on entity questions; the two regressions seen with the naive filters are gone. |
| **Narrative Flow** | Additive: contradictions stacked with "however", no chronology. | Analytical: explains the ON Semiconductor loss/price-surge contradiction via the buyback programme instead of listing both. |

---

### Summary

**Where metadata helped most.** The largest gain is at retrieval time on entity-specific
questions. Vector similarity over `TITLE + SUMMARY` has no reliable notion of *which company* a
question concerns: for the Microsoft query only 2 distinct articles out of 25 retrieved slots
mention Microsoft at all, and one of them — a ServiceNow story — occupies four slots by itself.
Filtering by ticker turned a five-word refusal into a ~160-word answer from 10 genuine `MSFT`
articles covering partnerships, the post-earnings share move, competitive pressure around
Amazon's Trainium chips, and OpenAI's move to reduce its dependence on Microsoft. The Amazon
query improved for the same reason: v1 answered with a UPS headline, an Nvidia roundup and a
Woodward note, while v2.1 answers from Amazon's own $100bn AI investment and the AWS CEO
interview.

At generation time the gain is smaller but real: citations went from 0 in every v1 answer to
3–12 per answer. That does not change the correctness of the facts; it changes the reader's
ability to verify them.

**The two regressions found earlier are fixed, and the ablation proves it.** Both were real
results from the first version of the filters, which is why `detect_filters_naive` is retained as
an ablation arm rather than deleted:

1. *Over-matching.* The token `financial` matched the industry *Financial Data & Stock Exchanges*,
   so any question containing "financial news" was silently filtered into that industry — the
   inflation answer ended up confined to the FTSE 100 and UK CPI. With `GENERIC_INDUSTRY_TOKENS`,
   the inflation and investor-sentiment queries are correctly left unfiltered, and the inflation
   answer now covers the Fed's May meeting, tariff-driven grocery prices at Dollar Tree, the macro
   outlook and the rotation into gold — ~198 words with 4 citations against v1's ~60 with none.
2. *Constraints intersected with AND.* The Amazon query detected `TICKER = AMZN` **and**
   `INDUSTRY = Financial Data & Stock Exchanges`; no article satisfies both, so the filter returned
   0 documents and only the fallback prevented an empty answer. With precedence
   (ticker ▸ industry ▸ sector) the query retrieves 10 `AMZN` documents and **no fallback is
   triggered anywhere in the run** — the comparison table shows `FELLBACK_V20 = 1` against
   `FELLBACK_V21 = 0`.

**Two limitations that filtering cannot fix.**

*Taxonomy boundaries are not question boundaries.* "The energy industry" in ordinary usage covers
renewables and utilities; `SECTOR = Energy` in the GICS taxonomy does not. The v2.1 energy answer
is a well-cited account of oil and gas — falling prices, OPEC+ output, tariffs, drilling budgets —
but it has lost the renewable-subsidy repeal and the Xcel Energy wildfire litigation that v1
surfaced. More focused, and less complete as an answer to the question a human asked. No blocklist
repairs this; only a broader notion of topical scope would. The retail filter shows the mirror
image: `REIT - Retail` (shopping-centre landlords, not retailers) is matched alongside the five
genuine retail industries. It did no visible damage here, but it is the same class of error in the
permissive direction.

*Filtering fixes the entity, not the topic.* The v2.1 Amazon answer is unambiguously about Amazon,
but it drifts toward AI investment rather than the automation and logistics that were asked about,
because within the `AMZN` slice ranking is still pure similarity. Getting the right company is a
precondition for a good answer, not a guarantee of one.

**Metadata quality is a dependency, not a given.** Of the 490 unique tickers in the corpus, 11
(2.2%) come back from Yahoo Finance with no name, sector or industry: ANSS, BK, CTRA, DAY, FI,
HOLX, IPG, JNPR, K, MMC and WBA. Crucially, `yf.Ticker(...).info` does **not** raise on these — it
logs a 404 and returns a dict with missing keys — so the original `try/except` never fired and the
gap was invisible. Falling back from `longName` to `shortName` recovered several tickers that
previously had a sector and industry but no name at all (UPS, RF and FDX among them), which is why
651 name keys are available for matching. Those 11 remaining tickers cannot be reached by company
name, only by symbol.

**Overall.** Metadata is worth adding, and the corrected filters improve every question relative to
the naive ones. But the design principle the evidence supports is that **metadata should bias
retrieval rather than gate it**: boosting the similarity of documents matching a detected ticker or
sector, instead of excluding everything else, would keep the Microsoft and Amazon gains, avoid
needing a fallback at all, and would not have amputated the renewables coverage from the energy
answer.